<a href="https://colab.research.google.com/github/lab-rasool/SIIM/blob/main/notebooks/SIIM_LocalLLMs_Backup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SIIM 2026 Learning Lab — Backup Cloud Instance
### Running Local LLMs Behind Institutional Firewalls (LL4022)

This notebook is a **fallback** for the hands-on labs. If a laptop won't
cooperate during the workshop, run these cells top to bottom to get:

- **Ollama** serving an open model (Lab 1)
- **OpenWebUI** — a private, ChatGPT-style interface, reachable at a public URL (Lab 2)
- the **Lab 3 clinical workflows** (radiology summarization + pathology extraction)

Modeled on [Axenide/Open-WebUI-Colab](https://github.com/Axenide/Open-WebUI-Colab).

---

> ## ⚠️ Read this first — this is the *opposite* of "behind the firewall"
> A Colab instance is a **public cloud machine**. It is the right tool for a
> conference demo with **synthetic data**, and the wrong tool for anything real.
>
> **Use synthetic data only. Never paste real patient data or PHI into this
> notebook or the public OpenWebUI URL it creates.** The whole point of the
> Learning Lab is that the production pattern keeps the model *inside* your
> network — this backup deliberately steps outside it, so treat everything
> here as public.
>
> The public URL is unauthenticated until you create an OpenWebUI account in
> the browser. Shut the runtime down (Runtime → Disconnect and delete runtime)
> when you're done so the tunnel closes.

## Step 0 — Pick your model(s)
`llama3.2` matches Lab 1 (~2 GB, fast). Set `LARGE_MODEL` to also pull a
bigger model for the side-by-side comparison in Lab 3 — leave it blank to skip.

**Tip:** use a GPU runtime (Runtime → Change runtime type → T4 GPU) so the
models run quickly. Ollama uses the GPU automatically when one is present.

In [ ]:
# Workshop model configuration
SMALL_MODEL = "llama3.2"        # Lab 1 default (~2 GB)
LARGE_MODEL = "qwen2.5:7b"     # for the small-vs-large comparison; set to "" to skip

import os
os.environ["SMALL_MODEL"] = SMALL_MODEL
os.environ["LARGE_MODEL"] = LARGE_MODEL
print("Will pull:", SMALL_MODEL, "and" , LARGE_MODEL or "(no large model)")

## Step 1 — Install Ollama and pull the model(s)
Installs the Ollama runtime, starts it in the background, and pulls your
model(s). On a T4 GPU this takes a couple of minutes.

**Use a GPU runtime before running this:** Runtime → Change runtime type →
**T4 GPU**. Ollama uses the GPU automatically *once it can detect it* — the
cell below installs the detection tools (`pciutils`, `lshw`) so it can. On a
CPU runtime `llama3.2` still works but is slow, and a 7B model is impractical
for a live demo (set `LARGE_MODEL = ""` in Step 0).

In [ ]:
# Ollama's installer ships zstd-compressed archives and detects GPUs via
# lspci/lshw — a fresh Colab runtime has none of these, so install them first.
# Without zstd the install fails; without pciutils/lshw it silently falls back
# to CPU even on a GPU runtime.
!sudo apt-get update -qq
!sudo apt-get install -y -qq zstd pciutils lshw

# Install the Ollama runtime
!curl -fsSL https://ollama.com/install.sh | sh

# Make sure the binary actually landed before we try to run it
import shutil, subprocess, time, os
ollama_bin = shutil.which('ollama') or '/usr/local/bin/ollama'
assert os.path.exists(ollama_bin), (
    'Ollama did not install. Re-run this cell; if it persists, check that '
    'the zstd install above succeeded.'
)

# Is a GPU actually attached? If this errors, you're on a CPU runtime:
# Runtime -> Change runtime type -> T4 GPU, then re-run from this cell.
# (llama3.2 still works on CPU; a 7B model will be very slow.)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'No GPU detected — running on CPU.'

# Colab has no systemd, so start the server ourselves in the background
subprocess.Popen([ollama_bin, 'serve'])
time.sleep(8)

# Pull the workshop model(s)
!ollama pull $SMALL_MODEL
if os.environ.get("LARGE_MODEL"):
    get_ipython().system('ollama pull $LARGE_MODEL')

# Confirm what's running locally (Lab 1, step 3)
!curl -s localhost:11434/api/tags | python3 -m json.tool

## Step 2 — Install OpenWebUI
OpenWebUI needs Python 3.11, so we install it into its own virtual
environment (Colab's default Python may differ). This cell only *installs* —
we start the servers a couple of cells down.

In [ ]:
# Install OpenWebUI into a Python 3.11 venv at an ABSOLUTE path, so it can
# be found no matter what the current directory is when later cells run.
!sudo apt-get update -qq
!sudo apt-get install -y -qq python3.11 python3.11-venv python3.11-dev

!python3.11 -m venv /content/venv
!/content/venv/bin/python -m pip install --upgrade pip -q
!/content/venv/bin/pip install open-webui -q
print("OpenWebUI installed at /content/venv")

## Step 3 — Clone the workshop repo
Brings in the Lab 3 scripts and the **synthetic** clinical datasets.

In [ ]:
import os, glob

# Always use an absolute repo path and only clone if it isn't already there.
# (Re-running this cell will NOT create nested SIIM/SIIM folders.)
REPO_DIR = "/content/SIIM"
if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    !rm -rf {REPO_DIR}
    !git clone https://github.com/lab-rasool/SIIM.git {REPO_DIR}

# Find where the lab scripts actually live — repo root or a subfolder —
# so this works however the code was pushed to the repo.
hits = glob.glob(os.path.join(REPO_DIR, '**', 'summarize_report.py'), recursive=True)
LAB_DIR = os.path.dirname(hits[0]) if hits else REPO_DIR
os.environ['LAB_DIR'] = LAB_DIR
%cd {LAB_DIR}
print('Lab scripts dir:', LAB_DIR)
!ls -1

## Step 4 — Start OpenWebUI and get a public URL
Ollama is already running from Step 1. This starts OpenWebUI on port 8081
and opens a public tunnel with [Pinggy](https://pinggy.io) (no account needed).

When the cell prints a URL like `https://....pinggy.link`, open it, create a
local OpenWebUI account, and you've got a private ChatGPT-style interface that
already sees the model(s) you pulled.

Keep this cell running — closing it closes the tunnel. (Alternatives if Pinggy
is blocked on conference Wi-Fi: `cloudflared` or an `ngrok` token.)

In [ ]:
# Ollama is already serving from Step 1. Start OpenWebUI in the background
# using ABSOLUTE paths (re-running earlier cells can't break this), then open
# the public tunnel. Logs go to a file so the URL below stays readable.
import subprocess, time

log = open('/content/openwebui.log', 'w')
subprocess.Popen(['/content/venv/bin/open-webui', 'serve', '--port', '8081'],
                 stdout=log, stderr=log)
print('Starting OpenWebUI… first boot can take ~30-60s.')
time.sleep(40)

# Public tunnel via Pinggy (no account). KEEP THIS CELL RUNNING — closing it
# closes the tunnel. Look for a line like https://xxxx.pinggy.link
!echo '--- Public URL below ---'; \
  ssh -o StrictHostKeyChecking=no -p 443 -R0:localhost:8081 qr@a.pinggy.io

## Step 5 — Run the Lab 3 clinical workflows in the cloud
These hit the same local Ollama running in this runtime. Run this in a
**separate** cell while Step 4 keeps the tunnel alive (use Runtime → Run
after, or just run the cells below). All data is synthetic.

In [ ]:
# Radiology — structured 3-line impressions
!python3 $LAB_DIR/summarize_report.py --model $SMALL_MODEL \
    --report $LAB_DIR/data/radiology/ct_chest_001.txt

In [ ]:
# Pathology — free text -> validated structured JSON
!python3 $LAB_DIR/extract_pathology.py --model $SMALL_MODEL \
    --report $LAB_DIR/data/pathology/path_lung_002.txt

In [ ]:
# Small vs. larger model, side by side (needs LARGE_MODEL set in Step 0)
import os
if os.environ.get('LARGE_MODEL'):
    get_ipython().system('python3 $LAB_DIR/summarize_report.py --compare $SMALL_MODEL $LARGE_MODEL --report $LAB_DIR/data/radiology/mri_brain_003.txt')
else:
    print('Set LARGE_MODEL in Step 0 to run the comparison.')